# TP 3 : Vision par ordinateur
**Module :** Technologies IA — ENSA Fès — ILIA2
**Étudiant(e) :** *(votre nom)*

> Ce notebook est prévu pour **Google Colab** (GPU gratuit recommandé : *Exécution > Modifier le type d'exécution > GPU*).
> Images à téléverser avant de lancer les exercices : `image1.jpg`, `image_objects.jpg`, `people.jpg`, `ref.jpg`, `test.jpg`.

## 0. Installation et fonctions utilitaires

In [ ]:
# Bibliothèques de base + YOLO
!pip install -q opencv-python numpy matplotlib ultralytics

# dlib (version précompilée) + face_recognition sans recompiler dlib
!pip install -q dlib-bin
!pip install -q face_recognition --no-deps
!pip install -q face_recognition_models Click Pillow

In [ ]:
import os, time
import cv2
import numpy as np
import matplotlib.pyplot as plt

def show(img, title="", gray=False, size=(8, 6)):
    """Affiche une image OpenCV (BGR) avec matplotlib (fonctionne en Colab et en local)."""
    plt.figure(figsize=size)
    if gray or img.ndim == 2:
        plt.imshow(img, cmap="gray")
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

In [ ]:
# Téléverser les images (Colab). En local, placez simplement les fichiers à côté du notebook.
try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    print("Hors Colab : placez vos images dans le dossier courant.")

---
## Partie 1 : Concepts de base de la vision par ordinateur
### Exercice 1 : Analyse d'une image

In [ ]:
# 1-2. Lecture et affichage
img = cv2.imread("image1.jpg")
assert img is not None, "image1.jpg introuvable : téléversez-la d'abord."
print("Type :", type(img), "| Forme (H, L, canaux) :", img.shape, "| dtype :", img.dtype)
show(img, "Image originale")

In [ ]:
# 3. Niveaux de gris
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
print("Forme en gris :", gray.shape)
show(gray, "Image en niveaux de gris", gray=True)

In [ ]:
# 4. Détection de contours (Canny)
edges = cv2.Canny(img, 100, 200)
show(edges, "Contours détectés (Canny)", gray=True)

# Variante : gris + flou gaussien avant Canny (question 3)
blur = cv2.GaussianBlur(gray, (5, 5), 0)
edges_blur = cv2.Canny(blur, 100, 200)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(edges, cmap="gray");      ax[0].set_title("Canny direct");          ax[0].axis("off")
ax[1].imshow(edges_blur, cmap="gray"); ax[1].set_title("Gris + flou + Canny");    ax[1].axis("off")
plt.show()

### Réponses aux questions (Partie 1)

**1. Que représente la matrice retournée par OpenCV pour l'image ?**
C'est un tableau NumPy à 3 dimensions `(hauteur, largeur, canaux)` de type `uint8`. Chaque case contient l'intensité (0 à 255) d'un canal de couleur d'un pixel. Attention : OpenCV utilise l'ordre **BGR** (et non RGB). Une image en niveaux de gris est une matrice 2D `(hauteur, largeur)`.

**2. Pourquoi convertit-on souvent l'image en niveaux de gris avant traitement ?**
- On passe de 3 canaux à 1 : **moins de données**, donc des calculs ~3 fois plus rapides et moins de mémoire.
- La plupart des algorithmes (détection de contours, de visages Haar/HOG, extraction de caractéristiques) reposent sur les **variations d'intensité** et non sur la couleur.
- Cela rend le traitement moins sensible aux variations de couleur d'éclairage.

**3. Que peut nous apporter le filtre gaussien `cv2.GaussianBlur(gray, (5, 5), 0)` ?**
Il **lisse** l'image en faisant une moyenne pondérée (poids gaussiens) des pixels voisins : il **réduit le bruit** et les petits détails parasites. Avant Canny, cela évite de détecter de faux contours dus au bruit et donne des contours plus propres et plus continus. Plus le noyau est grand, plus le flou est fort (et plus les détails fins sont perdus).

---
## Partie 2 : Traitement d'images et vidéos avec OpenCV
### Exercice 2 : Lecture vidéo / webcam

Sous Colab, la webcam locale et `cv2.imshow` ne sont pas disponibles. Le code ci-dessous est donc fourni en deux versions :
1. **version locale** (webcam, touche `q` pour quitter) à exécuter dans un script Python / Jupyter local ;
2. **version fichier vidéo** (fonctionne aussi sur Colab) qui convertit en niveaux de gris et **enregistre** la vidéo traitée.

In [ ]:
# --- Version locale : webcam en niveaux de gris, touche 'q' pour quitter ---
# Mettre RUN_WEBCAM = True uniquement si vous exécutez ce notebook sur votre PC.
RUN_WEBCAM = False

if RUN_WEBCAM:
    cap = cv2.VideoCapture(0)          # 0 = webcam ; ou chemin d'une vidéo
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        cv2.imshow("Webcam (gris)", gray_frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()

In [ ]:
# --- Version fichier vidéo : niveaux de gris + enregistrement (marche aussi sur Colab) ---
def video_en_gris(source, sortie="sortie_gris.mp4", afficher=False, max_frames=None):
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        raise IOError(f"Impossible d'ouvrir la source : {source}")
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 20.0
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(sortie, fourcc, fps, (w, h), isColor=False)

    n = 0
    while True:
        ret, frame = cap.read()
        if not ret:                      # fin de la vidéo
            break
        gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        out.write(gray_frame)
        if afficher:
            cv2.imshow("Enregistrement", gray_frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        n += 1
        if max_frames and n >= max_frames:
            break
    cap.release(); out.release()
    if afficher:
        cv2.destroyAllWindows()
    print(f"{n} images traitées -> {sortie}")

# Création d'une petite vidéo de test (si vous n'avez pas de vidéo) à partir de image1.jpg
def creer_video_test(chemin="video_test.mp4", nb=60):
    base = cv2.imread("image1.jpg")
    h, w = base.shape[:2]
    out = cv2.VideoWriter(chemin, cv2.VideoWriter_fourcc(*"mp4v"), 20.0, (w, h))
    for i in range(nb):
        M = np.float32([[1, 0, i * 2], [0, 1, 0]])      # petit déplacement horizontal
        out.write(cv2.warpAffine(base, M, (w, h)))
    out.release()
    return chemin

if not os.path.exists("video.mp4"):
    VIDEO = creer_video_test()
else:
    VIDEO = "video.mp4"
video_en_gris(VIDEO)

### Réponse à la question (Partie 2)

**Que se passe-t-il si on utilise une vidéo enregistrée à la place de la webcam ?**
Il suffit de remplacer `VideoCapture(0)` par le chemin du fichier : `VideoCapture("video.mp4")`. Différences :
- la vidéo a une **fin** : `cap.read()` renvoie `ret = False` et la boucle s'arrête d'elle-même (une webcam, elle, tourne indéfiniment) ;
- la cadence est celle du fichier (`CAP_PROP_FPS`), alors qu'avec `waitKey(1)` l'affichage peut aller **plus vite** que la vitesse réelle ; il faut utiliser `waitKey(1000/fps)` pour une lecture à vitesse normale ;
- on peut connaître le nombre total d'images (`CAP_PROP_FRAME_COUNT`) et se déplacer dans la vidéo, ce qui est impossible avec un flux webcam ;
- le traitement est **reproductible** (mêmes images à chaque exécution).

---
## Partie 3 : Détection et segmentation d'objets avec Detectron2
### Exercice 3 : Détection d'objets (Mask R-CNN)

In [ ]:
# Installation de Detectron2 (Colab, quelques minutes)
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

In [ ]:
import torch
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog

CONFIG = "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"

def creer_predictor(seuil=0.5):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(CONFIG))
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = seuil
    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(CONFIG)
    if not torch.cuda.is_available():
        cfg.MODEL.DEVICE = "cpu"
    return cfg, DefaultPredictor(cfg)

def detecter(img_bgr, seuil=0.5):
    cfg, predictor = creer_predictor(seuil)
    outputs = predictor(img_bgr)
    meta = MetadataCatalog.get(cfg.DATASETS.TRAIN[0])
    v = Visualizer(img_bgr[:, :, ::-1], meta, scale=1.2)
    out = v.draw_instance_predictions(outputs["instances"].to("cpu"))
    return out.get_image()[:, :, ::-1], outputs, meta

img_obj = cv2.imread("image_objects.jpg")
assert img_obj is not None, "image_objects.jpg introuvable."
res, outputs, meta = detecter(img_obj, seuil=0.5)
show(res, "Mask R-CNN - seuil 0.5", size=(10, 7))

# Q1 : liste des objets détectés
classes = outputs["instances"].pred_classes.cpu().numpy()
scores = outputs["instances"].scores.cpu().numpy()
for c, s in zip(classes, scores):
    print(f"{meta.thing_classes[c]:<15} {s:.2f}")

In [ ]:
# Q2 : effet du seuil SCORE_THRESH_TEST
for seuil in [0.3, 0.5, 0.7, 0.9]:
    res, outputs, meta = detecter(img_obj, seuil=seuil)
    n = len(outputs["instances"])
    show(res, f"Seuil = {seuil}  ->  {n} objet(s) détecté(s)", size=(8, 6))

### Réponses aux questions (Partie 3)

**1. Quels objets ont été détectés ?**
La cellule précédente affiche la liste exacte (classe COCO + score de confiance). *(Recopier ici le résultat obtenu avec votre image : par exemple `person`, `tie`, `bird`...)*. On observe aussi que le modèle, entraîné sur les 80 classes de COCO, peut se tromper sur des sujets hors COCO : un tigre peut être étiqueté `zebra`, un chien `cat`, etc.

**2. Effet du seuil `SCORE_THRESH_TEST` :**
- **Seuil bas (0.3)** : plus d'objets détectés, mais plus de **faux positifs** (étiquettes erronées, doublons, objets inexistants).
- **Seuil moyen (0.5)** : bon compromis.
- **Seuil élevé (0.7 - 0.9)** : seules les détections très sûres sont gardées ; les faux positifs disparaissent mais on risque de **manquer** des objets réels (faux négatifs).
Le seuil règle donc le compromis **précision / rappel**.

---
## Partie 4 : Détection et reconnaissance faciale avec Dlib
### Exercice 4 : Détection faciale

In [ ]:
import dlib

detector = dlib.get_frontal_face_detector()
img = cv2.imread("people.jpg")
assert img is not None, "people.jpg introuvable."
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

faces = detector(gray)
print("Nombre de visages détectés :", len(faces))
for face in faces:
    x, y, w, h = face.left(), face.top(), face.width(), face.height()
    cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)

show(img, "Visages détectés (dlib HOG)", size=(10, 7))

### Exercice 5 : Reconnaissance faciale

In [ ]:
import face_recognition
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"   # dlib sur CPU

image_ref = face_recognition.load_image_file("ref.jpg")
image_test = face_recognition.load_image_file("test.jpg")

enc_ref_list = face_recognition.face_encodings(image_ref)
enc_test_list = face_recognition.face_encodings(image_test)
print("Visages dans ref.jpg :", len(enc_ref_list), "| dans test.jpg :", len(enc_test_list))

assert enc_ref_list, "Aucun visage trouvé dans ref.jpg"
assert enc_test_list, "Aucun visage trouvé dans test.jpg"

encodage_ref = enc_ref_list[0]
encodage_test = enc_test_list[0]

result = face_recognition.compare_faces([encodage_ref], encodage_test, tolerance=0.6)
distance = face_recognition.face_distance([encodage_ref], encodage_test)[0]
print("Même personne ?", result[0], f"(distance = {distance:.3f}, seuil = 0.6)")

In [ ]:
# Q1 : image test avec plusieurs visages -> on compare TOUS les visages trouvés avec la référence
for i, enc in enumerate(enc_test_list):
    d = face_recognition.face_distance([encodage_ref], enc)[0]
    print(f"Visage {i} : distance = {d:.3f} -> {'même personne' if d <= 0.6 else 'personne différente'}")

### Réponses aux questions (Exercice 5)

**1. Que se passe-t-il si l'image test contient plusieurs visages ?**
`face_encodings()` renvoie **une liste** avec un encodage (vecteur de 128 valeurs) par visage. Avec `[0]` on ne compare que le **premier** visage détecté, ce qui peut donner un mauvais résultat. Il faut parcourir tous les encodages (cellule ci-dessus) et garder la plus petite distance. Si aucun visage n'est détecté, la liste est vide et `[0]` provoque une `IndexError`, d'où le test préalable.

**2. Quelle précision attendre en conditions réelles ?**
Le modèle de `face_recognition` annonce environ **99,4 %** sur le benchmark LFW, mais ce chiffre est obtenu sur des images de bonne qualité. En conditions réelles la précision baisse nettement avec : un **éclairage** faible ou contrasté, des **angles** de vue non frontaux, des visages partiellement cachés (lunettes, masque, main), une basse résolution, le vieillissement, des expressions fortes. Le choix du seuil de tolérance (0.6 par défaut) règle le compromis entre faux positifs et faux négatifs ; il faut le valider sur ses propres données. Il existe aussi des biais selon les populations représentées dans les données d'entraînement.

### Exercice 6 : Points caractéristiques du visage (landmarks)

In [ ]:
img = cv2.imread("people.jpg")
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)     # face_recognition attend du RGB
landmarks_list = face_recognition.face_landmarks(rgb)
print("Visages :", len(landmarks_list))

for landmarks in landmarks_list:
    for partie, points in landmarks.items():
        for (x, y) in points:
            cv2.circle(img, (x, y), 2, (0, 0, 255), -1)

show(img, "Landmarks du visage", size=(10, 7))
print("Parties retournées :", list(landmarks_list[0].keys()) if landmarks_list else "aucune")

In [ ]:
# Q2 : Eye Aspect Ratio (EAR)
from math import dist

def ear(eye):
    """eye = liste de 6 points (x, y) dans l'ordre fourni par face_recognition."""
    A = dist(eye[1], eye[5])
    B = dist(eye[2], eye[4])
    C = dist(eye[0], eye[3])
    return (A + B) / (2.0 * C)

for i, lm in enumerate(landmarks_list):
    e_g, e_d = ear(lm["left_eye"]), ear(lm["right_eye"])
    print(f"Visage {i} : EAR gauche = {e_g:.2f} | EAR droit = {e_d:.2f} | moyenne = {(e_g + e_d) / 2:.2f}")

### Réponses aux questions (Exercice 6)

**1. Parties du visage retournées par `face_landmarks` :** un dictionnaire (68 points au total) avec les clés
`chin` (menton / contour), `left_eyebrow`, `right_eyebrow`, `nose_bridge`, `nose_tip`, `left_eye`, `right_eye`, `top_lip`, `bottom_lip`.

**2. EAR (Eye Aspect Ratio) :**
$$EAR = \frac{\|p_2 - p_6\| + \|p_3 - p_5\|}{2\,\|p_1 - p_4\|}$$
Les deux numérateurs sont les hauteurs de l'œil, le dénominateur sa largeur. Œil ouvert : EAR ≈ 0,25 – 0,35 ; œil fermé : EAR tend vers 0.
Pour détecter un **clignement** : l'EAR passe sous un seuil (≈ 0,2) pendant 2 à 3 images puis remonte.
Pour détecter la **somnolence** d'un conducteur : si l'EAR reste sous le seuil pendant une durée anormalement longue (ex. plus de 1 à 2 secondes, ou beaucoup de frames consécutives), on déclenche une alerte sonore. On peut aussi compter la fréquence de clignements par minute.

---
## Exercice 7 : Détection d'objets en temps réel avec YOLO

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")      # modèle léger, téléchargé automatiquement

image_path = "image_objects.jpg"
if not os.path.exists(image_path):
    print(f"'{image_path}' non trouvé. Utilisation de 'people.jpg' à la place.")
    image_path = "people.jpg"

try:
    results = model(image_path, conf=0.5)
    show(results[0].plot(), "YOLOv8n", size=(10, 7))
except Exception as e:
    print("Erreur lors de la détection :", e)

In [ ]:
# Q1 : comparaison des temps d'inférence YOLOv8n vs Mask R-CNN (même image)
img_cmp = cv2.imread(image_path)
N = 5

# YOLO (1 passage d'échauffement puis moyenne)
model(img_cmp, conf=0.5, verbose=False)
t0 = time.time()
for _ in range(N):
    model(img_cmp, conf=0.5, verbose=False)
t_yolo = (time.time() - t0) / N

# Mask R-CNN (Detectron2)
_, predictor = creer_predictor(0.5)
predictor(img_cmp)
t0 = time.time()
for _ in range(N):
    predictor(img_cmp)
t_mask = (time.time() - t0) / N

print(f"YOLOv8n   : {t_yolo * 1000:.0f} ms / image")
print(f"Mask R-CNN: {t_mask * 1000:.0f} ms / image")
print(f"Mask R-CNN est ~{t_mask / t_yolo:.1f} fois plus lent")

In [ ]:
# Q2 : YOLO avec masques de segmentation
model_seg = YOLO("yolov8n-seg.pt")
res_seg = model_seg(image_path, conf=0.5)
show(res_seg[0].plot(), "YOLOv8n-seg (segmentation)", size=(10, 7))
print("Masques disponibles :", res_seg[0].masks is not None)

### Réponses aux questions (Exercice 7)

**1. Temps d'inférence :** voir la cellule ci-dessus (les valeurs dépendent du GPU/CPU). En général YOLOv8n est **nettement plus rapide** (quelques ms sur GPU, plusieurs dizaines d'images/s) que Mask R-CNN R50-FPN (plusieurs dizaines à centaines de ms par image).

**2. YOLO fournit-il des masques ?** Le modèle `yolov8n.pt` ne donne que des **boîtes englobantes**. Le modèle `yolov8n-seg.pt` fournit en plus des **masques de segmentation d'instance** (`results[0].masks`).

**3. Quand préférer chaque modèle ?**

| | **YOLO** | **Mask R-CNN** |
|---|---|---|
| Vitesse | Très rapide, temps réel | Plus lent |
| Précision des contours | Correcte (version `-seg`) | Masques plus précis |
| Matériel | Léger, embarqué, webcam | GPU conseillé |
| À préférer pour | Vidéo, temps réel, comptage, surveillance, mobile | Analyse hors-ligne, imagerie médicale, tâches exigeant des masques très précis |

---
## Mini-projet : floutage automatique des visages
Le mini-projet (option 4) est fourni dans le fichier `flou_visages.py` (voir `README.md`).
Démonstration rapide ci-dessous :

In [ ]:
def flouter_visages(img_bgr, noyau=(51, 51)):
    sortie = img_bgr.copy()
    gris = cv2.cvtColor(sortie, cv2.COLOR_BGR2GRAY)
    for f in detector(gris):
        x1, y1 = max(f.left(), 0), max(f.top(), 0)
        x2, y2 = min(f.right(), sortie.shape[1]), min(f.bottom(), sortie.shape[0])
        zone = sortie[y1:y2, x1:x2]
        if zone.size:
            sortie[y1:y2, x1:x2] = cv2.GaussianBlur(zone, noyau, 0)
    return sortie

show(flouter_visages(cv2.imread("people.jpg")), "Visages floutés", size=(10, 7))